## Basic Decoder-Only Transformer Implementation

In [1]:
# Installing files
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-24 07:01:45--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2026-03-24 07:01:45 (139 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { c:i for i, c in enumerate(chars)}
itos = { i:c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[id] for id in ids])

In [6]:
print(encode("hii there"))
decode(encode('hii there'))

[46, 47, 47, 1, 58, 46, 43, 56, 43]


'hii there'

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
split = int(0.9 * len(data))
train = data[:split]
valid = data[split:]

In [9]:
block_size = 256
batch_size = 64
dropout = 0.2
n_embd = 384
eval_iters = 200
max_iters = 5000
eval_interval = 500
n_blocks = 6
n_heads = 6
head_size = 64
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [10]:
torch.manual_seed(1337)

def get_batch(split):
  assert split in ['train', 'valid'], "split is either train or valid"

  if split == "train":
    data = train
  else:
    data = valid

  idxs = torch.randint(0, len(data)-block_size, (batch_size, ))

  x = torch.stack([data[idx:idx+block_size] for idx in idxs])
  y = torch.stack([data[idx+1:idx+block_size+1] for idx in idxs])
  x, y = x.to(device), y.to(device)

  return x, y

In [11]:
xb, yb = get_batch("train")
for i, j in zip(xb[0].tolist(), yb[0].tolist()):
  print(f"{decode([i])} -> {decode([j])}")


 -> N
N -> o
o -> t
t ->  
  -> G
G -> l
l -> o
o -> u
u -> c
c -> e
e -> s
s -> t
t -> e
e -> r
r -> '
' -> s
s ->  
  -> d
d -> e
e -> a
a -> t
t -> h
h -> ,
, ->  
  -> n
n -> o
o -> r
r ->  
  -> H
H -> e
e -> r
r -> e
e -> f
f -> o
o -> r
r -> d
d -> '
' -> s
s ->  
  -> b
b -> a
a -> n
n -> i
i -> s
s -> h
h -> m
m -> e
e -> n
n -> t
t -> 


 -> N
N -> o
o -> t
t ->  
  -> G
G -> a
a -> u
u -> n
n -> t
t -> '
' -> s
s ->  
  -> r
r -> e
e -> b
b -> u
u -> k
k -> e
e -> s
s -> ,
, ->  
  -> n
n -> o
o -> r
r ->  
  -> E
E -> n
n -> g
g -> l
l -> a
a -> n
n -> d
d -> '
' -> s
s ->  
  -> p
p -> r
r -> i
i -> v
v -> a
a -> t
t -> e
e ->  
  -> w
w -> r
r -> o
o -> n
n -> g
g -> s
s -> ,
, -> 


 -> N
N -> o
o -> r
r ->  
  -> t
t -> h
h -> e
e ->  
  -> p
p -> r
r -> e
e -> v
v -> e
e -> n
n -> t
t -> i
i -> o
o -> n
n ->  
  -> o
o -> f
f ->  
  -> p
p -> o
o -> o
o -> r
r ->  
  -> B
B -> o
o -> l
l -> i
i -> n
n -> g
g -> b
b -> r
r -> o
o -> k
k -> e
e -> 


 -> A
A -> b
b -> o

In [12]:
class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embd, n_embd * 4),
      nn.ReLU(),
      nn.Linear(n_embd * 4, n_embd),
      nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)

In [13]:
# self attention intuition
B, T, C = 4, 8, 32
mask = torch.tril(torch.ones((T, T)))
weights = torch.zeros((T, T))
weights = weights.masked_fill(mask==0, float('-inf'))
weights = F.softmax(weights, dim=-1)

x = torch.randn((B, T, C))

result = weights @ x # (B, T, C)

In [14]:
# self attention
x = torch.randn((B, T, C))

q = nn.Linear(C, head_size)
k = nn.Linear(C, head_size)
v = nn.Linear(C, head_size)

Q = q(x) # (B, T, C)
K = k(x) # (B, T, C)
V = v(x) # (B, T, C)

mask = torch.tril(torch.ones((T, T)))
weights = Q @ K.transpose(-2, -1) # (B, T, T)
weights = weights.masked_fill(mask==0, float('-inf'))
# weights = F.softmax(weights, dim=-1)
# att_vals = weights @ V # (B, T, C)

weights[1]

tensor([[-1.6264,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-3.7380, -1.9545,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-8.7824,  0.4638, -0.6661,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-4.9468,  0.8954,  5.2167,  3.2214,    -inf,    -inf,    -inf,    -inf],
        [-0.0575,  1.1498,  0.5224,  0.2227,  0.1329,    -inf,    -inf,    -inf],
        [ 5.0909, -4.8054,  2.0067,  1.7886, -6.1597,  2.1450,    -inf,    -inf],
        [-0.0839, -0.9598, -0.4138, -1.5499, -0.1190,  2.3677, -0.8182,    -inf],
        [ 0.1512, -3.1559,  4.9492, -1.6088,  2.4410, -0.9103, -4.1910,  2.5627]],
       grad_fn=<SelectBackward0>)

In [15]:
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.q = nn.Linear(n_embd, head_size, bias=False)
    self.k = nn.Linear(n_embd, head_size, bias=False)
    self.v = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('mask', torch.tril(torch.ones((block_size, block_size))))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape

    Q = self.q(x) # (B, T, head_size)
    K = self.k(x) # (B, T, head_size)
    V = self.v(x) # (B, T, head_size)

    weights = (Q @ K.transpose(-2, -1)) * head_size**-0.5 # (B, T, T)
    weights = weights.masked_fill(self.mask[:T, :T]==0, float('-inf'))
    weights = F.softmax(weights, dim=-1)
    weights = self.dropout(weights)
    att_vals = weights @ V # (B, T, head_size)

    return att_vals

In [16]:
class MultiHeadAttention(nn.Module):
  def __init__(self, n_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
    self.proj = nn.Linear(n_heads * head_size, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    x = torch.cat([head(x) for head in self.heads], dim=-1) # (B, T, n_heads * head_size)
    x = self.proj(x) # (B, T, n_embd)
    x = self.dropout(x)
    return x

In [17]:
class Block(nn.Module):
  def __init__(self, n_embd, n_heads):
    super().__init__()
    self.sa = MultiHeadAttention(n_heads, n_embd//n_heads)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    # x -> (B, T, n_embd)
    x = x + self.sa(self.ln1(x)) # (B, T, n_embd)
    x = x + self.ffwd(self.ln2(x)) # (B, T, n_embd)
    return x

In [18]:
class LayerNorm():
  def __init__(self, dim, eps = 1e-5):
    # dim = n
    self.eps = eps
    self.gamma = torch.ones(dim) # (dim)
    self.beta = torch.zeros(dim) # (dim)

  def __call__(self, x):
    # x -> (n_rows, n_cols)
    mean = x.mean(dim=1, keepdim=True) # (n_rows, 1)
    var = x.var(dim=1, keepdim=True) # (n_rows, 1)
    x = (x - mean) / (var + self.eps)**0.5
    x = x * self.gamma + self.beta # ()
    return x


In [19]:
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, n_embd) # (vocab_size, n_embd)
    self.positional = nn.Embedding(block_size, n_embd) # (block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_heads=n_heads) for _ in range(n_blocks)])
    self.ln = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, x, targets = None):
    # x -> (B, T), targets -> (B, T)
    B, T = x.shape
    token_embd = self.embedding(x) # (B, T, n_embd)
    pos_embd = self.positional(torch.arange(T, device=device)) # (T, n_embd)
    x = token_embd + pos_embd # (B, T, n_embd)
    x = self.blocks(x) # (B, T, n_embd)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      logits = logits.view(B*T, vocab_size) # (B*T, vocab_size)
      targets = targets.view(B*T) # (B*T, )
      loss = F.cross_entropy(logits, targets) # scalar

    return logits, loss

  def generate(self, x, max_new_tokens=10):
    # x -> (B, T)
    for _ in range(max_new_tokens):
      cropped_x = x[:, -block_size:]
      logits, _ = self(cropped_x) # (B, T, vocab_size)
      """
        for each sample in the batch, look at the last character's logits and choose the next character.
        append the character at the end of each sample
      """
      logits = logits[:, -1, :] # (B, vocab_size)
      probs = F.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      x = torch.cat((x, next_token), dim=-1) # (B, T+1)

    return x

m = BigramLanguageModel()
m = m.to(device)

In [20]:
@torch.no_grad()
def compute_loss():
  # prediction -> (B, T, vocab_size)
  # actual -> (B, T)
  result = {}
  m.eval()

  for split in ['train', 'valid']:
    losses = torch.zeros((eval_iters,))
    for i in range(eval_iters):
      xb, yb = get_batch(split)
      _, loss = m(xb, yb)
      losses[i] = loss.item()
    result[split] = losses.mean()

  m.train()
  return result

In [21]:
# training
optimizer = optim.NAdam(m.parameters(), lr=learning_rate)

for i in range(max_iters):
  x, y = get_batch("train")

  # reset gradients
  optimizer.zero_grad(set_to_none=True)

  # forward pass
  logits, loss = m(x, y)

  # backward pass
  loss.backward()

  # update
  optimizer.step()

  # log statistics
  # print(f"Step {i+1} | loss = {loss}")
  if i % eval_interval == 0:
    losses = compute_loss()
    print(f"Step {i}:  Train loss: {losses['train']},  Val loss: {losses['valid']}")

Step 0:  Train loss: 4.081393718719482,  Val loss: 4.107217311859131
Step 500:  Train loss: 2.0202231407165527,  Val loss: 2.0967938899993896
Step 1000:  Train loss: 1.6324423551559448,  Val loss: 1.7990843057632446
Step 1500:  Train loss: 1.4605188369750977,  Val loss: 1.662948489189148
Step 2000:  Train loss: 1.3605200052261353,  Val loss: 1.5824573040008545
Step 2500:  Train loss: 1.294276475906372,  Val loss: 1.5404653549194336
Step 3000:  Train loss: 1.23392653465271,  Val loss: 1.515556812286377
Step 3500:  Train loss: 1.185865044593811,  Val loss: 1.5018362998962402
Step 4000:  Train loss: 1.1441386938095093,  Val loss: 1.4876501560211182
Step 4500:  Train loss: 1.0982182025909424,  Val loss: 1.4911233186721802


In [22]:
# Loss logs
# Single head attention: 2.1058027744293213
# Multi head attention: 1.5715726613998413

In [26]:
# generate output
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


Lord Angelo, is fresh as I dorset last.

DUCHESS OF YORK:
I know throw it thee speed with the law of Gaunt and
thy kinsman, until it receive.

LORD STER:
Then, that mayor kinsmal I will have you again the head,
Commend with him desertaked state hand, and lambs at you:
but not doned does me safe; 'tis we smown,
to dangled curse our greater, and Sicilia conducteted,
'Tis timeless fallingness, and Warwick's throans dreams,'
When is't in hand? see 'twen all's eagle, no girl:
Wister; part conduced dream'd make dead;'
I thank it is the like.

CAMILLIUS! Dold more of Asmal thou of the king.

LOIZSLY:
No man give Verona to pice death, something frother deny,
To live her body accentioning may most fittle reason,
And as double the sound to my wrong and your love,
A man that worst shall never the duke I reserable
Who it is not?

MONTAGUE:
With thee Joicess, someth come with the flight; and promphisons,
Vhilties that plish shbout linealthed with them
To bouring that be my spirit.

MAMILLIUS:
I ha

In [27]:
PATH = "gpt2.pt"
torch.save(m.state_dict(), PATH)